# Step2: Annotation and Malignancy

Advanced analysis notebook for clustering, cell-type annotation, and tumor malignancy evidence.

CNV is kept in the annotation notebook because malignant-cell calls and malignant-state refinement depend on CNV evidence.

Input: `data/processed/Step2-sce_preprocessed.h5ad`.  
Output: `data/processed/Step3-sce_annotated.h5ad`.


---

## 1. Setup

### 1.1 Imports


In [ ]:
import scanpy as sc
import numpy as np
import pandas as pd
import os
import scLucid as scl
import matplotlib as mpl
mpl.rcParams['pdf.fonttype'] = 42
mpl.rcParams['ps.fonttype'] = 42
import matplotlib.pyplot as plt
plt.rcParams['font.family'] = 'Arial'
import warnings
warnings.filterwarnings('ignore')

scl.setup_logging("INFO")
scl.set_figure_params(
    dpi=300,
    dpi_save=300,
    figsize=(6, 5),
    style="seaborn-v0_8",
    style_dict={'axes.grid': True},
    color_theme="default"
)


### 1.2 Paths


In [ ]:
# Define working directories
BASE_DIR = '/Users/luye/Library/Mobile Documents/com~apple~CloudDocs/Projects/Ongoing/202604JJH'
DATA_DIR = os.path.join(BASE_DIR, "1-DATA")
RESULTS_DIR = os.path.join(BASE_DIR, "2-OUTPUT")
os.makedirs(RESULTS_DIR, exist_ok=True)


### 1.3 Colors


In [ ]:
###** colors
color = {
    'PBS': '#BFCFE3',
    "R301":"#8B4C9D"
}


---

## 2. Data Loading

### 2.1 Load Preprocessed Data


In [ ]:
# --- Load Preprocessed Data ---
# Load preprocessed data
print("Loading preprocessed data...")
adata = sc.read_h5ad(os.path.join(DATA_DIR, "Step2-sce_preprocessed.h5ad"))
print(f"Dataset loaded: {adata.shape[0]} cells, {adata.shape[1]} genes")

print("\n---- Analysis Workflow ----")


In [ ]:
##====== Diagnostic UMAP before integration ======##
sc.pp.neighbors(adata, use_rep="X_pca", n_neighbors=15, n_pcs=40)
sc.tl.umap(adata, min_dist=0.2)

# 保存诊断用embedding，便于和整合后结果对比
adata.obsm["X_umap_pca"] = adata.obsm["X_umap"]


In [ ]:
sc.pl.embedding(adata, basis='X_umap_pca',
                color=['group', 'sampleID'],
                title=['Group', 'Sample'],
                size=10,
                ncols=2, wspace=0.5, hspace=0.3,
                show=False,frameon=False,
            )


---

## 3. Clustering

### 3.1 Visualize Embeddings


In [ ]:
adata.obs['group'] = pd.Categorical(
    adata.obs['group'], 
    categories=[
    'PBS','R301'
], 
    ordered=True
)


### 3.2 Sample-Group Crosstab


In [ ]:
ctab = pd.crosstab(adata.obs['sampleID'],adata.obs['group'],margins=True)
print(ctab)


### 3.3 Over-clustering


In [ ]:
# --- 3.2 Over-clustering ---
clustering_cfg = scl.al.ClusteringConfig(
    method="leiden",
    resolution=0.8,
    use_rep="X_pca",
    key_added="leiden_clusters" # Final cluster column name
)
adata = scl.al.cluster_cells(adata, config=clustering_cfg)


### 3.4 QC on UMAP


In [ ]:
sc.pl.embedding(adata, basis='X_umap_pca',
                color=['pct_counts_mt', 'doubletdetection_score','heuristic_confidence_score'],
                ncols=3, wspace=0.5, hspace=0.3,
                show=False,frameon=False,cmap='viridis'
            )


In [ ]:
sc.pl.embedding(adata, basis='X_umap_pca',
                color=['leiden_clusters'],
                size=9,
                legend_loc="on data",
                legend_fontsize=14,
                #ncols=3, wspace=0.5, hspace=0.3,
                show=False,frameon=False,
            )


### 3.5 Save


In [ ]:
adata.write(DATA_DIR+ '/Step3-sce_annotated.h5ad', compression='gzip')


---

## 4. Cell Type Annotation

### 4.1 Load Data


In [ ]:
adata = sc.read_h5ad(os.path.join(DATA_DIR, "Step3-sce_annotated.h5ad"))
adata


In [ ]:
import gc 
gc.collect()


### 4.2 Coarse Marker Filter


In [ ]:
# ==============================================================================
# Step 2: Marker discovery (Marker Discovery & Enrichment)
# ==============================================================================
print("\n##====== 2. Marker Discovery & Enrichment ======##")
# --- 4.2 Coarse marker filter ---
# No strict filtering yet，目的是为富集分析提供尽可能多的基因
de_config = scl.al.DifferentialConfig(
    groupby='leiden_clusters',
    use_raw=True,
    pval_cutoff=0.05
)
scl.al.find_markers(adata, config=de_config)


### 4.3 Fine Marker Filter


In [ ]:
# --- 4.3 Filter high-specificity markers ---
# 现在，我们应用严格的过滤来找到用于注释和可视化的“金标准”marker
filter_cfg = scl.al.FilterMarkersConfig(
    key="rank_genes_groups",              
    key_added="highly_specific_markers_df", # Final filtered result
    min_log2fc=0.5,
    max_padj=0.01,
    min_in_group_pct=0.2,
    max_out_group_pct=None,
    min_diff_pct=0.1, # Min 10% difference
    keep_top_n=50
)
highly_specific_markers_df = scl.al.filter_markers(adata, config=filter_cfg)


### 4.4 Visualize Markers


In [ ]:
# --- 4.4 Visualize markers ---
scl.al.visualize_markers(
    adata,
    markers=highly_specific_markers_df, # Pass filtered DataFrame
    groupby="leiden_clusters",
    plot_type="dotplot",
    swap_axes=True,
    n_genes_per_group=3 # Top 5 per cluster
)


### 4.5 Rank Gene Groups


In [ ]:
sc.pl.rank_genes_groups(adata, n_genes=25, sharey=False)


### 4.6 Enrichment


In [ ]:
# --- 4.5 Enrichment analysis ---
# Enrichment sensitivity note，使用过滤前的marker列表效果可能更好
enrichment_config = scl.al.EnrichmentConfig(
    de_key='rank_genes_groups_df', # Use default output
    mode='offline',
    organism='mouse',
    gene_sets_offline=['go_bp', 'reactome'] # 可以是一个列表
)
scl.al.run_enrichment(adata, groupby='leiden_clusters', config=enrichment_config)


### 4.7 Export Summary


In [ ]:
# --- 2.5 导出整合了“精筛Marker”和“富集结果”的摘要 ---
summary_path = os.path.join(RESULTS_DIR, "al_annotation/annotation_summary.md")
scl.al.summarize_markers_and_enrichment(
    adata,
    groupby="leiden_clusters",
    markers_df=highly_specific_markers_df, # Use filtered markers
    enrichment_key="enrichment",            # Auto-load from .uns
    n_markers=25,
    n_terms=10,
    summary_file=summary_path
)


### CNV Analysis


In [ ]:
# ==============================================================================
# CNV Analysis - use raw counts (CopyKAT-like pipeline)
# ==============================================================================

import os
import pandas as pd
import numpy as np
import scanpy as sc
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.sparse import issparse
from scipy.ndimage import uniform_filter1d

# ==============================================================================
# 预备工作：创建输出目录
# 请确保 RESULTS_DIR, DATA_DIR 和 adata 在外部已定义
# ==============================================================================
CNV_DIR = os.path.join(RESULTS_DIR, "cnv_analysis")
os.makedirs(CNV_DIR, exist_ok=True)

print("="*80)
print("Starting CopyKAT-like CNV analysis")
print(f"Results will be saved to: {CNV_DIR}")
print("="*80)

# ==============================================================================
# Step 0: 重新读取原始count数据
# ==============================================================================
print("\nStep 0: Loading original count data...")

# 请根据你的实际情况修改路径
raw_data_path = os.path.join(DATA_DIR, 'Step1-sce_cleaned.h5ad')  # 修改为你的实际路径
print(f"Reading from: {raw_data_path}")

if not os.path.exists(raw_data_path):
    print(f"ERROR: File not found: {raw_data_path}")
    print("\nPlease provide the path to your original count data.")
    raise FileNotFoundError(f"Raw count data not found at {raw_data_path}")

adata_counts = sc.read_h5ad(raw_data_path)
print(f"Loaded raw counts: {adata_counts.shape[0]} cells × {adata_counts.shape[1]} genes")

# 验证数据是否为原始counts
print("\nVerifying data type...")
if issparse(adata_counts.X):
    data_sample = adata_counts.X[:100, :100].toarray()
else:
    data_sample = adata_counts.X[:100, :100]

max_val = data_sample.max()
has_decimals = np.any(data_sample % 1 != 0)

if max_val < 20 or has_decimals:
    print("\n⚠️  WARNING: Data appears to be normalized/log-transformed!")
    print("Attempting to find counts in layers...")
    if 'counts' in adata_counts.layers:
        print("Found 'counts' layer, using it...")
        adata_counts.X = adata_counts.layers['counts'].copy()
    elif 'raw_counts' in adata_counts.layers:
        print("Found 'raw_counts' layer, using it...")
        adata_counts.X = adata_counts.layers['raw_counts'].copy()
    else:
        raise ValueError("Cannot proceed without raw count data. Please provide the original count matrix.")
else:
    print("✓ Data appears to be raw counts")

# ==============================================================================
# Step 1: 匹配细胞并筛选
# ==============================================================================
print("\nStep 1: Matching cells with processed data...")
current_cells = adata.obs.index  # adata是你当前处理到的数据

common_cells = adata_counts.obs.index.intersection(current_cells)
print(f"Common cells: {len(common_cells)} / {len(current_cells)}")

if len(common_cells) == 0:
    raise ValueError("Cell barcodes don't match. Please check data sources.")

adata_counts = adata_counts[common_cells, :].copy()

print("\nTransferring metadata from processed data...")
metadata_cols = ['leiden_clusters', 'sampleID', 'cell_type']  # 根据需要修改
for col in metadata_cols:
    if col in adata.obs.columns:
        adata_counts.obs[col] = adata.obs.loc[common_cells, col]
        print(f"  Transferred: {col}")

# ==============================================================================
# Step 2: 添加基因组坐标
# ==============================================================================
print("\nStep 2: Adding genomic coordinates...")
gene_info_path = '/Users/luye/Data/Sig/gene_info_mmu.txt'  # 请确保此路径正确
gene_info = pd.read_csv(gene_info_path, sep='\t')

gene_info = gene_info.rename(columns={
    'chromosome_name': 'chromosome',
    'start_position': 'start',
    'end_position': 'end',
    'external_gene_name': 'gene_name'
})

gene_info = gene_info[gene_info['gene_biotype'] == 'protein_coding'].copy()
gene_info = gene_info.drop_duplicates(subset=['gene_name'], keep='first')
gene_info = gene_info.set_index('gene_name')

adata_counts.var['chromosome'] = gene_info.reindex(adata_counts.var_names)['chromosome']
adata_counts.var['start'] = gene_info.reindex(adata_counts.var_names)['start']
adata_counts.var['end'] = gene_info.reindex(adata_counts.var_names)['end']

initial_genes = adata_counts.n_vars
adata_counts = adata_counts[:, adata_counts.var[['chromosome', 'start', 'end']].notna().all(axis=1)].copy()
print(f"Genes with genomic coordinates: {adata_counts.n_vars} / {initial_genes}")

adata_counts.var['chromosome'] = adata_counts.var['chromosome'].astype(str)
standard_chroms = [str(i) for i in range(1, 20)] + ['X', 'Y']
adata_counts = adata_counts[:, adata_counts.var['chromosome'].isin(standard_chroms)].copy()
print(f"Standard chromosomes: {adata_counts.n_vars} genes")

# ==============================================================================
# Step 3: 基因过滤 - 基于原始counts
# ==============================================================================
print("\nStep 3: Filtering genes based on expression...")
if issparse(adata_counts.X):
    gene_counts = np.array(adata_counts.X.sum(axis=0)).flatten()
    cells_per_gene = np.array((adata_counts.X > 0).sum(axis=0)).flatten()
else:
    gene_counts = adata_counts.X.sum(axis=0)
    cells_per_gene = (adata_counts.X > 0).sum(axis=0)

min_cells = int(0.05 * adata_counts.n_obs)
gene_filter = (cells_per_gene >= min_cells) & (gene_counts >= min_cells)
adata_counts = adata_counts[:, gene_filter].copy()

chr_counts = adata_counts.var['chromosome'].value_counts()
valid_chroms = chr_counts[chr_counts >= 100].index.tolist()
adata_counts = adata_counts[:, adata_counts.var['chromosome'].isin(valid_chroms)].copy()

adata_counts.var['chromosome'] = pd.Categorical(
    adata_counts.var['chromosome'],
    categories=[str(i) for i in range(1, 20)] + ['X', 'Y'],
    ordered=True
)
adata_counts = adata_counts[:, adata_counts.var.sort_values(['chromosome', 'start']).index].copy()
print(f"Final gene count after filtering: {adata_counts.n_vars}")

# ==============================================================================
# Step 4: 定义参考细胞 (Helper robust MAD)
# ==============================================================================
def mad(x, axis=0, eps=1e-6):
    med = np.median(x, axis=axis)
    mad_val = np.median(np.abs(x - np.expand_dims(med, axis=axis)), axis=axis)
    return mad_val + eps

# ==============================================================================
# Step 5: 选择参考细胞群
# ==============================================================================
print("\n[CNV] Step 5: Defining reference cells...")
REF_CLUSTERS = ['6','7','8','9','13','14','15','17','20','21','22','23']  # T and NK clusters in your data

if 'leiden_clusters' not in adata_counts.obs.columns:
    raise ValueError("leiden_clusters not found in adata_counts.obs")

leiden_str = adata_counts.obs['leiden_clusters'].astype(str)
ref_mask = leiden_str.isin(REF_CLUSTERS).values
n_ref = ref_mask.sum()
if n_ref < 200:
    print(f"⚠️  Reference cells too few ({n_ref}). Consider adding more immune clusters.")

adata_counts.obs['cnv_is_reference'] = ref_mask
print(f"[CNV] Reference cells: {n_ref} / {adata_counts.n_obs} ({n_ref/adata_counts.n_obs*100:.1f}%)")

# ==============================================================================
# Step 6: 过滤线粒体/核糖体/细胞周期等易产生伪影的基因
# ==============================================================================
print("\n[CNV] Step 6: Filtering genes prone to artifacts (mt/ribo/cell-cycle)...")
symbol_cols = [c for c in ['gene_name', 'gene_symbol', 'symbol', 'external_gene_name'] if c in adata_counts.var.columns]

if len(symbol_cols) > 0:
    sym_col = symbol_cols[0]
    gene_symbols = adata_counts.var[sym_col].astype(str)
else:
    # 强制将 var_names 转换为 Pandas Series，以保持后续方法返回类型的一致性
    gene_symbols = pd.Series(adata_counts.var_names, index=adata_counts.var_names).astype(str)

mt_mask = gene_symbols.str.match(r'^(mt-|MT-|Mt-)', na=False)
ribo_mask = gene_symbols.str.match(r'^(Rpl|Rps|RPL|RPS)', na=False)

cell_cycle_genes = set()  
cc_mask = gene_symbols.isin(cell_cycle_genes) if len(cell_cycle_genes) > 0 else np.zeros(adata_counts.n_vars, dtype=bool)

# 使用 np.asarray() 替代 .values，实现安全的类型转换和按位运算
drop_mask = (np.asarray(mt_mask) | np.asarray(ribo_mask) | np.asarray(cc_mask))

adata_counts.var['cnv_mt'] = np.asarray(mt_mask)
adata_counts.var['cnv_ribo'] = np.asarray(ribo_mask)
adata_counts.var['cnv_cc'] = np.asarray(cc_mask)

print(f"[CNV] Dropping mt: {np.sum(mt_mask)}, ribo: {np.sum(ribo_mask)}, cell-cycle: {np.sum(cc_mask)}")
adata_cnv = adata_counts[:, ~drop_mask].copy()

# ==============================================================================
# Step 7: 标准化 Counts (log1p CPM-like)
# ==============================================================================
print("\n[CNV] Step 7: Normalizing counts...")
sc.pp.normalize_total(adata_cnv, target_sum=1e4)
sc.pp.log1p(adata_cnv)

X = adata_cnv.X
if issparse(X):
    X = X.toarray()
else:
    X = X.copy()

# ==============================================================================
# Step 8: 染色体级别的平滑处理 (Vectorized)
# ==============================================================================
print("\n[CNV] Step 8: Chromosome-wise smoothing (vectorized)...")
chromosomes = adata_cnv.var['chromosome'].astype(str).values
unique_chroms = pd.unique(chromosomes)

window_size = 101
X_smooth = np.zeros_like(X, dtype=np.float32)

for chrom in unique_chroms:
    idx = np.where(chromosomes == chrom)[0]
    if len(idx) < 30:
        X_smooth[:, idx] = X[:, idx]
        continue
    X_chr = X[:, idx]
    X_chr_smooth = uniform_filter1d(X_chr, size=window_size, axis=1, mode='nearest')
    X_smooth[:, idx] = X_chr_smooth

# ==============================================================================
# Step 9: 匹配参考细胞进行稳健缩放 (MAD)
# ==============================================================================
print("\n[CNV] Step 9: Centering to reference and robust scaling (MAD)...")
ref_mask_cnv = adata_cnv.obs['cnv_is_reference'].values
if ref_mask_cnv.sum() < 50:
    raise ValueError("Too few reference cells after filtering.")

X_ref = X_smooth[ref_mask_cnv, :]
ref_mean = X_ref.mean(axis=0)
ref_mad = mad(X_ref, axis=0)

Z = (X_smooth - ref_mean) / ref_mad
Z = np.clip(Z, -10, 10)

# ==============================================================================
# Step 10: 计算 CNV Scores
# ==============================================================================
print("\n[CNV] Step 10: Computing robust CNV scores...")
cnv_chr_scores = {}
for chrom in unique_chroms:
    idx = np.where(chromosomes == chrom)[0]
    if len(idx) < 30:
        continue
    cnv_chr_scores[f'chr{chrom}_score'] = np.median(np.abs(Z[:, idx]), axis=1)

cnv_chr_df = pd.DataFrame(cnv_chr_scores, index=adata_cnv.obs_names)

adata_cnv.obs['cnv_score'] = cnv_chr_df.median(axis=1).values
adata_cnv.obs['cnv_extreme_frac'] = (np.abs(Z) > 2.0).mean(axis=1)

for c in cnv_chr_df.columns:
    adata_cnv.obs[c] = cnv_chr_df[c].values

# ==============================================================================
# Step 11: 非整倍体预测 (Conservative threshold + optional GMM)
# ==============================================================================
print("\n[CNV] Step 11: Predicting aneuploid cells (conservative)...")
ref_scores = adata_cnv.obs.loc[ref_mask_cnv, 'cnv_score'].values
ref_med = np.median(ref_scores)
ref_mad_score = np.median(np.abs(ref_scores - ref_med)) + 1e-6

thr = ref_med + 3.0 * ref_mad_score
adata_cnv.obs['predicted_class_thr'] = np.where(adata_cnv.obs['cnv_score'] > thr, 'aneuploid', 'diploid')

print(f"[CNV] Reference cnv_score median={ref_med:.4f}, MAD={ref_mad_score:.4f}, threshold={thr:.4f}")

try:
    from sklearn.mixture import GaussianMixture
    feats = adata_cnv.obs[['cnv_score', 'cnv_extreme_frac']].values
    gmm = GaussianMixture(n_components=2, random_state=42)
    gmm_labels = gmm.fit_predict(feats)
    means = [adata_cnv.obs['cnv_score'].values[gmm_labels == i].mean() for i in [0,1]]
    aneu_comp = int(np.argmax(means))
    adata_cnv.obs['predicted_class_gmm'] = np.where(gmm_labels == aneu_comp, 'aneuploid', 'diploid')
except Exception as e:
    print("[CNV] GMM skipped due to:", repr(e))

adata_cnv.obs['predicted_class'] = adata_cnv.obs['predicted_class_thr']

# ==============================================================================
# Step 12: 简单的分布统计并保存 CSV
# ==============================================================================
print("\n[CNV] Summary by sample and cluster...")
if 'sampleID' in adata_cnv.obs.columns:
    sample_tab = (adata_cnv.obs
                  .groupby('sampleID')['predicted_class']
                  .value_counts(normalize=True)
                  .rename('frac')
                  .reset_index())
    print(sample_tab.pivot(index='sampleID', columns='predicted_class', values='frac').fillna(0).round(3))

adata_cnv.obs.to_csv(os.path.join(CNV_DIR, "cnv_predictions_improved.csv"))

# ==============================================================================
# Step 13: Visualizations (revised & improved with percentages)
# ==============================================================================
print("\nStep 13: Generating visualizations (revised)...")

# 1) CNV score distribution
print("\n1) CNV score distribution...")
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(adata_cnv.obs.loc[ref_mask_cnv, 'cnv_score'], bins=50,
             alpha=0.6, label='Reference', color='#3498db', density=True)
axes[0].hist(adata_cnv.obs.loc[~ref_mask_cnv, 'cnv_score'], bins=50,
             alpha=0.6, label='Non-reference', color='#e74c3c', density=True)
axes[0].axvline(thr, color='black', linestyle='--', linewidth=2, label=f'Thr={thr:.3f}')
axes[0].set_xlabel('CNV Score')
axes[0].set_ylabel('Density')
axes[0].set_title('CNV Score: Reference vs Others')
axes[0].legend()
axes[0].grid(alpha=0.3)

for pred_class, color in [('diploid', '#2ecc71'), ('aneuploid', '#e74c3c')]:
    mask = (adata_cnv.obs['predicted_class'] == pred_class).values
    if mask.sum() == 0: continue
    axes[1].hist(adata_cnv.obs.loc[mask, 'cnv_score'], bins=50,
                 alpha=0.6, label=f'{pred_class} (n={mask.sum()})', density=True, color=color)
axes[1].axvline(thr, color='black', linestyle='--', linewidth=2, label=f'Thr={thr:.3f}')
axes[1].set_xlabel('CNV Score')
axes[1].set_ylabel('Density')
axes[1].set_title('CNV Score by Predicted Class')
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.tight_layout()
fig_file = os.path.join(CNV_DIR, "cnv_score_distribution.pdf")
plt.savefig(fig_file, dpi=300, bbox_inches='tight')
plt.show()

# 2) UMAP overlays
print("\n2) UMAP overlays...")
if 'X_umap' not in adata_cnv.obsm.keys():
    if 'X_umap' in adata.obsm.keys() and np.all(adata_cnv.obs_names == adata.obs_names):
        adata_cnv.obsm['X_umap'] = adata.obsm['X_umap'].copy()
    else:
        print("⚠️  Computing UMAP on adata_cnv (may not match your main embedding).")
        sc.pp.pca(adata_cnv, n_comps=50)
        sc.pp.neighbors(adata_cnv)
        sc.tl.umap(adata_cnv)

sc.pl.umap(
    adata_cnv,
    color=['cnv_score', 'cnv_extreme_frac', 'predicted_class'],
    wspace=0.4,
    show=True
)

# 3) Per-chromosome CNV scores (box/violin)
print("\n3) CNV scores per chromosome...")
chr_score_cols = [c for c in adata_cnv.obs.columns if c.startswith('chr') and c.endswith('_score')]
if len(chr_score_cols) > 0:
    df = adata_cnv.obs[chr_score_cols + ['predicted_class']].copy()
    long = df.melt(id_vars='predicted_class', var_name='chrom', value_name='cnv_chr_score')
    long['chrom'] = (long['chrom'].str.replace('chr', '', regex=False).str.replace('_score', '', regex=False))
    
    chrom_order = [str(i) for i in range(1, 20)] + ['X', 'Y']
    long['chrom'] = pd.Categorical(long['chrom'], categories=chrom_order, ordered=True)
    long = long.sort_values('chrom')

    fig, ax = plt.subplots(figsize=(16, 5))
    sns.violinplot(
        data=long, x='chrom', y='cnv_chr_score', hue='predicted_class',
        split=True, inner='quartile', palette={'diploid': '#2ecc71', 'aneuploid': '#e74c3c'}, ax=ax, cut=0
    )
    ax.set_xlabel('Chromosome')
    ax.set_ylabel('Median(|Z|) per chromosome')
    ax.set_title('Per-chromosome CNV score by predicted class')
    ax.grid(axis='y', alpha=0.3)
    plt.tight_layout()
    plt.savefig(os.path.join(CNV_DIR, "cnv_scores_per_chromosome_violin.pdf"), dpi=300, bbox_inches='tight')
    plt.show()

# 4) Aneuploid proportion by sample & cluster
print("\n4) Aneuploid proportion by sample and cluster...")
def aneu_stats(group_col):
    if group_col not in adata_cnv.obs.columns: return None
    tab = adata_cnv.obs.groupby(group_col)['predicted_class'].value_counts(normalize=False).unstack(fill_value=0)
    tab['total'] = tab.sum(axis=1)
    if 'aneuploid' not in tab.columns: tab['aneuploid'] = 0
    tab['aneuploid_pct'] = tab['aneuploid'] / tab['total'] * 100
    return tab.sort_values('aneuploid_pct', ascending=True)

sample_stats = aneu_stats('sampleID')
cluster_stats = aneu_stats('leiden_clusters')

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

if sample_stats is not None:
    top = sample_stats
    axes[0].barh(top.index.astype(str), top['aneuploid_pct'], color='#e74c3c', height=0.6)
    axes[0].set_xlabel('Aneuploid %')
    axes[0].set_title('Aneuploid proportion by sample')
    max_val = top['aneuploid_pct'].max()
    axes[0].set_xlim(0, max_val * 1.15 if max_val > 0 else 100)
    axes[0].grid(axis='x', alpha=0.3)
    for i, v in enumerate(top['aneuploid_pct']):
        axes[0].text(v + (max_val * 0.01), i, f"{v:.1f}%", va='center', color='black', fontsize=10, fontweight='bold')
else:
    axes[0].set_axis_off(); axes[0].set_title("No sampleID in adata_cnv.obs")

if cluster_stats is not None:
    top = cluster_stats.tail(25)
    axes[1].barh(top.index.astype(str), top['aneuploid_pct'], color='#e74c3c', height=0.6)
    axes[1].set_xlabel('Aneuploid %')
    axes[1].set_title('Aneuploid proportion by cluster (top 25)')
    max_val = top['aneuploid_pct'].max()
    axes[1].set_xlim(0, max_val * 1.15 if max_val > 0 else 100)
    axes[1].grid(axis='x', alpha=0.3)
    for i, v in enumerate(top['aneuploid_pct']):
        axes[1].text(v + (max_val * 0.01), i, f"{v:.1f}%", va='center', color='black', fontsize=10, fontweight='bold')
else:
    axes[1].set_axis_off(); axes[1].set_title("No leiden_clusters in adata_cnv.obs")

plt.tight_layout()
plt.savefig(os.path.join(CNV_DIR, "aneuploid_proportion_by_group.pdf"), dpi=300, bbox_inches='tight')
plt.show()

# 5) CNV heatmap along genome
print("\n5) CNV heatmap along genome (sampled cells)...")
np.random.seed(42)
n_show = min(800, adata_cnv.n_obs)
show_cells = np.random.choice(adata_cnv.obs_names, size=n_show, replace=False)

show_obs = adata_cnv.obs.loc[show_cells].copy()
show_obs['is_ref'] = show_obs.index.map(lambda x: adata_cnv.obs.loc[x, 'cnv_is_reference'])
show_obs = show_obs.sort_values(['is_ref', 'cnv_score'], ascending=[False, True])
show_cells = show_obs.index.values

cell_idx = adata_cnv.obs_names.get_indexer(show_cells)
Z_show = Z[cell_idx, :]

fig, ax = plt.subplots(figsize=(18, 7))
im = ax.imshow(Z_show, aspect='auto', cmap='bwr', vmin=-3, vmax=3, interpolation='nearest')
ax.set_xlabel('Genes ordered by chromosome & position')
ax.set_ylabel('Sampled cells (ref first, then low→high cnv_score)')
ax.set_title('CNV Z heatmap (centered to reference, MAD-scaled)')
cbar = plt.colorbar(im, ax=ax, fraction=0.02, pad=0.01)
cbar.set_label('Z')

chrom = np.array(chromosomes)
boundaries = np.where(chrom[:-1] != chrom[1:])[0] + 0.5
for b in boundaries:
    ax.axvline(b, color='k', linewidth=0.3, alpha=0.4)

plt.tight_layout()
plt.savefig(os.path.join(CNV_DIR, "cnv_heatmap_sampled.pdf"), dpi=300, bbox_inches='tight')
plt.show()

print("\n" + "="*80)
print("CNV visualization completed!")
print("="*80)


In [ ]:
# ==============================================================================
# Transfer results to adata
# ==============================================================================
print("\nTransferring CNV results back to original adata...")

# Transfer CNV results to original adata
cnv_cols = ['cnv_score', 'predicted_class']
#cnv_cols += [col for col in adata_norm.obs.columns if col.startswith('chr') and col.endswith('_score')]

for col in cnv_cols:
    adata.obs[col] = adata_cnv.obs.loc[adata.obs.index, col]

print(f"Transferred {len(cnv_cols)} CNV columns to original adata")

print("\nCNV Analysis Summary:")
print("="*80)
print(f"Total cells analyzed: {adata.n_obs}")
print(f"Predicted aneuploid cells: {(adata.obs['predicted_class'] == 'aneuploid').sum()}")
print(f"Predicted diploid cells: {(adata.obs['predicted_class'] == 'diploid').sum()}")
print(f"Mean CNV score: {adata.obs['cnv_score'].mean():.4f}")
print("="*80)

print("\nCNV analysis completed!")
print(f"All results saved to: {CNV_DIR}")


In [ ]:
adata.write(DATA_DIR+ '/Step3-sce_annotated.h5ad', compression='gzip')


### 4.8 Manual Refinement


In [ ]:
# Define mapping file path
mapping_file_path = os.path.join(RESULTS_DIR, "al_annotation/manual_mapping.xlsx")

# Call mapping function
scl.al.apply_annotation_mapping(
    adata,
    cluster_key="leiden_clusters",       # Specify cluster column
    mapping=mapping_file_path,           # Pass file path
    key_added="celltype"         # Define new column name
)
print("Annotation complete:")
print(adata.obs['celltype'].value_counts())


In [ ]:
#adata = adata[~adata.obs["leiden_clusters"].isin(["4","10","22"]), :]
adata


### 4.9 Set Categorical Order


In [ ]:
adata.obs['group'] = adata.obs['group'].cat.reorder_categories(['PBS','R301'], ordered=True)


In [ ]:
celltype_colors = {
    # Malignant epithelial/tumor states
    "Mesenchymal-like malignant cells": "#B2182B",
    "Growth factor-high malignant cells": "#D6604D",
    "Translation-high malignant cells": "#F4A582",
    "Hypoxic/glycolytic malignant cells": "#CA0020",
    "Cycling malignant cells": "#8B0000",
    "IFN-stimulated malignant cells": "#E78AC3",
    "Proliferating malignant cells": "#67001F",
    "CDH13+ mesenchymal-like malignant cells": "#A50F15",

    # Myeloid cells
    "Neutrophils": "#FDB863",
    "IL1B+ inflammatory macrophages": "#2166AC",
    "CCL2+ inflammatory macrophages": "#4393C3",
    "C1q+ antigen-presenting macrophages": "#92C5DE",
    "MRC1+ resident-like macrophages": "#053061",
    "CTSK+ osteoclast-like macrophages": "#5E3C99",

    # Lymphoid cells
    "Cytotoxic NK/NKT-like cells": "#1B9E77",
    "Cytotoxic CD8 T cells": "#66A61E",
    "Activated T cells": "#B2DF8A",
    "B cells": "#7570B3",

    # Dendritic cells
    "pDCs": "#E6AB02",
    "CLEC9A+ cDCs": "#A6761D",
    "CCR7+ mature DCs": "#FFD92F",

    # Stromal / vascular / mast
    "ECM-remodeling fibroblasts": "#A65628",
    "Vascular endothelial cells": "#4DAF4A",
    "Mast cells": "#F781BF"
}
celltype_order = [
    # Malignant compartment
    "Mesenchymal-like malignant cells",
    "Growth factor-high malignant cells",
    "Translation-high malignant cells",
    "Hypoxic/glycolytic malignant cells",
    "Cycling malignant cells",
    "IFN-stimulated malignant cells",
    "Proliferating malignant cells",
    "CDH13+ mesenchymal-like malignant cells",

    # Myeloid compartment
    "Neutrophils",
    "IL1B+ inflammatory macrophages",
    "CCL2+ inflammatory macrophages",
    "C1q+ antigen-presenting macrophages",
    "MRC1+ resident-like macrophages",
    "CTSK+ osteoclast-like macrophages",

    # T / NK / B cells
    "Cytotoxic NK/NKT-like cells",
    "Cytotoxic CD8 T cells",
    "Activated T cells",
    "B cells",

    # DCs
    "pDCs",
    "CLEC9A+ cDCs",
    "CCR7+ mature DCs",

    # Stromal / vascular / mast
    "ECM-remodeling fibroblasts",
    "Vascular endothelial cells",
    "Mast cells"
]


In [ ]:
# Set categorical order
adata.obs['celltype'] = adata.obs['celltype'].astype('category')
adata.obs['celltype'] = adata.obs['celltype'].cat.reorder_categories(celltype_order, ordered=True)


### 4.10 Marker Gene Analysis


In [ ]:
# Calculate cluster markers
# Use t-test or Wilcoxon test
sc.tl.rank_genes_groups(adata, groupby='celltype', method="wilcoxon", use_raw=True)


In [ ]:
sc.pl.rank_genes_groups_dotplot(
    adata, groupby="celltype", standard_scale="var", n_genes=5, key="rank_genes_groups"
)


In [ ]:
sc.pl.rank_genes_groups(adata, n_genes=25, sharey=False)


In [ ]:
marker_dict = {
    # -------------------------
    # Malignant cell states
    # -------------------------
    "Mesenchymal-like malignant cells": [
        "Cdh13", "Slit2", "Sema3a", "Hmga2", "Prrx1", "Vcam1", "Col1a2"
    ],

    "Growth factor-high malignant cells": [
        "Ereg", "Hbegf", "Grem1", "Timp1", "Nes", "Itga6", "Cald1"
    ],

    "Translation-high malignant cells": [
        "Eif4a1", "Eif3a", "Eif3e", "Eef1a1", "Eef2", "Rplp0", "Rps3", "Rps6"
    ],

    "Hypoxic/glycolytic malignant cells": [
        "P4ha2", "Plod2", "Ankrd37", "Bnip3", "Ndrg1", "Pdk1", "Higd1a", "Eno2"
    ],

    "Cycling malignant cells": [
        "Mki67", "Top2a", "Prc1", "Aurkb", "Cenpf", "Kif11", "Ube2c"
    ],

    "IFN-stimulated malignant cells": [
        "Ifit1", "Ifit3", "Usp18", "Isg15", "Oasl2", "Gbp2", "Rsad2", "Ifih1"
    ],

    "Proliferating malignant cells": [
        "Cenpf", "Ube2c", "Cdc20", "Plk1", "Ccnb1", "Aurka", "Tpx2", "Kif20a"
    ],

    "CDH13+ mesenchymal-like malignant cells": [
        "Cdh13", "Slit2", "Sema3a", "Zfpm2", "Nav2", "Ghr", "Hmga2"
    ],


    # -------------------------
    # Myeloid cells
    # -------------------------
    "Neutrophils": [
        "S100a8", "S100a9", "Csf3r", "Trem1", "Cxcl2", "Il1b", "G0s2", "Hdc"
    ],

    "IL1B+ inflammatory macrophages": [
        "Il1b", "Ccr2", "Cd14", "Itgam", "Cd86", "Clec7a", "Cybb", "Lyz2"
    ],

    "CCL2+ inflammatory macrophages": [
        "Ccl2", "Ccl6", "Ccl7", "Ccl9", "Cxcl2", "Mafb", "Cd68", "Mpeg1"
    ],

    "C1q+ antigen-presenting macrophages": [
        "C1qa", "C1qb", "C1qc", "Mrc1", "Apoe", "Ctss", "Lgmn", "Trem2"
    ],

    "MRC1+ resident-like macrophages": [
        "Mrc1", "Gas6", "F13a1", "Apoe", "Selenop", "Stab1", "Csf1r", "Colec12"
    ],

    "CTSK+ osteoclast-like macrophages": [
        "Ctsk", "Acp5", "Atp6v0d2", "Mmp9", "Tnfrsf11a", "Nfatc1", "Mmp14", "Lpl"
    ],


    # -------------------------
    # Lymphoid cells
    # -------------------------
    "Cytotoxic NK/NKT-like cells": [
        "Ncr1", "Nkg7", "Gzmb", "Prf1", "Klrd1", "Ctsw", "Il2rb", "Txk"
    ],

    "Cytotoxic CD8 T cells": [
        "Cd8a", "Cd8b1", "Cd3d", "Cd3e", "Trac", "Pdcd1", "Lag3", "Nkg7", "Prf1"
    ],

    "Activated T cells": [
        "Icos", "Cd2", "Cd3d", "Cd3e", "Cd28", "Cd247", "Itk", "Lck"
    ],

    "B cells": [
        "Cd79a", "Cd79b", "Ms4a1", "Pax5", "Ebf1", "Ighd", "Ighm", "Igkc"
    ],


    # -------------------------
    # Dendritic cells
    # -------------------------
    "pDCs": [
        "Siglech", "Ccr9", "Tcf4", "Irf8", "Lair1", "Rnase6", "Tmem229b"
    ],

    "CLEC9A+ cDCs": [
        "Clec9a", "Wdfy4", "Flt3", "Irf8", "Cd24a", "H2-Aa", "H2-Eb1", "Cd74"
    ],

    "CCR7+ mature DCs": [
        "Ccr7", "Ccl22", "Flt3", "Cd83", "Mreg", "Lsp1", "Itga4", "Plxnc1"
    ],


    # -------------------------
    # Stromal / vascular / mast cells
    # -------------------------
    "ECM-remodeling fibroblasts": [
        "Col1a1", "Col3a1", "Col5a2", "Col6a2", "Col6a3", "Fbn1", "Mmp2", "Loxl2"
    ],

    "Vascular endothelial cells": [
        "Emcn", "Ptprb", "Esam", "Egfl7", "Cdh5", "Flt1", "Ramp2", "Sparcl1"
    ],

    "Mast cells": [
        "Cma1", "Mrgprb2", "Ms4a2", "Mcpt4", "Tpsb2", "Fcer1a", "Cpa3", "Hdc"
    ]
}


In [ ]:
sc.tl.dendrogram(adata,'celltype',use_rep="X_pca")
with plt.rc_context({'figure.figsize':(6,5)}):
    sc.pl.dotplot(
        adata,
        groupby="celltype",
        var_names=marker_dict,
        dendrogram=False,
        cmap='RdBu_r', #viridis ,
        #categories_order = order,
        standard_scale="var",  # standard scale: normalize each gene to range from 0 to 1
        save=True
        )


In [ ]:
genes = ["Ptprc","Col1a1","Epcam","Krt8","Krt18","Krt19","Tacstd2"]
df = sc.get.obs_df(adata, keys=genes, use_raw=True)
for g in genes:
    print(g, "pct>0:", (df[g]>0).mean(), "mean:", df[g].mean(), "p99:", np.quantile(df[g], 0.99))


In [ ]:
sc.pl.embedding(
    adata, 
    basis='X_umap_pca',
    color=["Ptprc", "Pecam1", "Dcn","Hmga2"],
    cmap='RdBu_r',
    ncols=4,
    vmax=3,vmin=0,
    show=True,frameon=False,legend_loc=None,
    save=False
)


### 4.11 Final refinement


In [ ]:
adata


In [ ]:
#adata.obs['celltype'] = adata.obs['celltype'].replace('Folr2+ TAMs', 'Ccl8+ TAMs')


In [ ]:
sc.pl.embedding(adata, basis='X_umap_pca',
                color=['celltype'],
                title = ['All cells'],
                #ncols=2, wspace=0.45,
                #legend_loc="on data",
                legend_fontsize=9,
                size=8,
                palette=celltype_colors,
                show=True, 
                frameon=False,
                save='UMAP_anno_all.pdf')


In [ ]:
# ----------------------------
# Config
# ----------------------------
outdir = "./figures"
os.makedirs(outdir, exist_ok=True)

UMAP_BASIS = "X_umap_pca"   # 你已有这个
SCORE_COL = "cnv_score"
STATUS_COL = "predicted_class"  # CNV status column
GROUP_COL = "group"
CELLTYPE_COL = "celltype"
SAMPLE_COL = "sampleID"         # 如果你要 sample-level 统计（推荐）
treatment_order =["PBS","R301"]
# Optional: set order if you have it
GROUP_ORDER = treatment_order if "treatment_order" in globals() else None

# ----------------------------
# Guardrails
# ----------------------------
missing = [c for c in [SCORE_COL, STATUS_COL, GROUP_COL, CELLTYPE_COL] if c not in adata.obs.columns]
if missing:
    raise KeyError(f"Missing columns in adata.obs: {missing}")

if SCORE_COL not in adata.obs.columns or adata.obs[SCORE_COL].isna().all():
    raise ValueError(f"{SCORE_COL} not found or all NA in adata.obs")

# DataFrame for seaborn
df = adata.obs[[SCORE_COL, STATUS_COL, GROUP_COL, CELLTYPE_COL] + ([SAMPLE_COL] if SAMPLE_COL in adata.obs.columns else [])].copy()

# keep only finite numeric scores
df[SCORE_COL] = pd.to_numeric(df[SCORE_COL], errors="coerce")
df = df.loc[np.isfinite(df[SCORE_COL])].copy()

# apply categorical order for group if provided
if GROUP_ORDER is not None:
    df[GROUP_COL] = pd.Categorical(df[GROUP_COL], categories=GROUP_ORDER, ordered=True)

# ----------------------------
# 1) UMAP overview: cnv_score + predicted_class
# ----------------------------
fig, axes = plt.subplots(1, 2, figsize=(14, 5), constrained_layout=True)

sc.pl.embedding(
    adata,
    basis=UMAP_BASIS,
    color=SCORE_COL,
    ax=axes[0],
    show=False,
    title="CNV Score (CopyKAT-like pipeline)",
    cmap="RdBu_r",
    vmin=0, vmax=2
)

sc.pl.embedding(
    adata,
    basis=UMAP_BASIS,
    color=STATUS_COL,
    ax=axes[1],
    show=False,
    title="CNV Analysis Status"
)

fig.savefig(os.path.join(outdir, "cnv_umap_overview.pdf"), dpi=300, bbox_inches="tight")
plt.show()
plt.close(fig)

# ----------------------------
# 2) Boxplot: CNV scores by treatment group (cell-level)
# ----------------------------
fig, ax = plt.subplots(figsize=(10, 6))

sns.boxplot(
    data=df,
    x=GROUP_COL,
    y=SCORE_COL,
    order=GROUP_ORDER,
    palette=color if "group_colors" in globals() else None,
    ax=ax
)
# optional: add points (downsample if huge)
# sns.stripplot(data=df.sample(min(5000, len(df)), random_state=0),
#               x=GROUP_COL, y=SCORE_COL, order=GROUP_ORDER,
#               color="black", size=1.5, alpha=0.25, ax=ax)

ax.set_title("CNV Score by Treatment Group (cell-level)")
ax.set_xlabel("")
ax.set_ylabel("CNV Score")
ax.tick_params(axis="x", rotation=45)
for lab in ax.get_xticklabels():
    lab.set_ha("right")

fig.tight_layout()
fig.savefig(os.path.join(outdir, "cnv_by_group_celllevel.pdf"), dpi=300, bbox_inches="tight")
plt.show()
plt.close(fig)

# ----------------------------
# 3) Boxplot: CNV scores by celltype (cell-level)
# ----------------------------
fig, ax = plt.subplots(figsize=(12, 6))

sns.boxplot(
    data=df,
    x=CELLTYPE_COL,
    y=SCORE_COL,
    palette=celltype_colors if "celltype_colors" in globals() else None,
    ax=ax
)

ax.set_title("CNV Score by Cell Type (cell-level)")
ax.set_xlabel("")
ax.set_ylabel("CNV Score")
ax.tick_params(axis="x", rotation=45)
for lab in ax.get_xticklabels():
    lab.set_ha("right")

fig.tight_layout()
fig.savefig(os.path.join(outdir, "cnv_by_celltype_celllevel.pdf"), dpi=300, bbox_inches="tight")
plt.show()
plt.close(fig)

# ----------------------------
# 4) (推荐) Sample-level summary to avoid pseudo-replication
#     每个 sample 聚合成一个值，再做箱线图
# ----------------------------
if SAMPLE_COL in df.columns:
    # per-sample mean (you can switch to median)
    df_s = (df.groupby([GROUP_COL, SAMPLE_COL], observed=True)[SCORE_COL]
              .mean()
              .reset_index(name="cnv_score_mean"))

    # keep order
    if GROUP_ORDER is not None:
        df_s[GROUP_COL] = pd.Categorical(df_s[GROUP_COL], categories=GROUP_ORDER, ordered=True)

    fig, ax = plt.subplots(figsize=(10, 6))

    sns.boxplot(
        data=df_s,
        x=GROUP_COL,
        y="cnv_score_mean",
        order=GROUP_ORDER,
        palette=color if "group_colors" in globals() else None,
        ax=ax
    )
    sns.stripplot(
        data=df_s,
        x=GROUP_COL,
        y="cnv_score_mean",
        order=GROUP_ORDER,
        color="black",
        size=4,
        jitter=0.18,
        alpha=0.75,
        ax=ax
    )

    ax.set_title("CNV Score by Treatment Group (sample-level mean)")
    ax.set_xlabel("")
    ax.set_ylabel("Mean CNV Score (per sample)")
    ax.tick_params(axis="x", rotation=45)
    for lab in ax.get_xticklabels():
        lab.set_ha("right")

    fig.tight_layout()
    fig.savefig(os.path.join(outdir, "cnv_by_group_samplelevel_mean.pdf"), dpi=300, bbox_inches="tight")
    plt.show()
    plt.close(fig)


### 4.12 UMAP Comparison


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 5))
axes = axes.flatten()

for idx, (ax, group) in enumerate(zip(axes, treatment_order)):
    sc.pl.embedding(
        adata[adata.obs['group'] == group],
        basis='X_umap_pca',
        color=["celltype"],
        title=group,
        size=16,
        palette=celltype_colors,
        show=False,
        frameon=False,
        ax=ax,
        legend_loc='on data' if idx == 0 else None,
        legend_fontsize=8,
        legend_fontoutline=2
    )

plt.tight_layout()
fig.savefig(os.path.join(RESULTS_DIR, 'UMAP_2groups_comparison.pdf'),
            bbox_inches='tight', dpi=300)
plt.show()
print(f"Saved to: {os.path.join(RESULTS_DIR, 'UMAP_groups_comparison.pdf')}")


### 4.13 Distribution


In [ ]:
ctab = pd.crosstab(adata.obs['celltype'],adata.obs['group'],margins=True)
print(ctab)


### 4.14 Save


In [ ]:
adata.write(DATA_DIR+ '/Step3-sce_annotated.h5ad', compression='gzip')


### 4.15 Export Metadata


In [ ]:
meta = pd.DataFrame(data=adata.obs)
meta.to_csv(RESULTS_DIR + '/sce_metadata_all.csv',sep="\t") 
